### Input Requirements

> 🚨 **Structure Files**

- `NODE_FILE` *(.xyz, required)*  
  File containing the node fragment (higher-connectivity building unit).  
  **Location:** `0_node/` directory  
  **Constraint:** Exactly one `.xyz` file must be present in this directory.

- `LINKER_FILE` *(.xyz, required)*  
  File containing the linker fragment (lower-connectivity building unit).  
  **Location:** `0_linker/` directory  
  **Constraint:** Exactly one `.xyz` file must be present in this directory.

#### Connection Definition

- `DUMMY_ATOMS` *(required)*  
  Dummy atoms must be placed at all intended connection points (i.e., positions where bonds between node and linker will be formed).  

  - **He** → single bond  
  - **Se** → double bond  

  **Constraint:** Dummy atom types must be consistent with the selected `BOND_TYPE` parameter.

In [1]:
import coflandscaper as cl

/Users/gregorlauter/Documents/PhD/2d-cof-mapper/COF-Landscaper/.venv/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


### General Settings & Options

The following parameters control structure generation and output behavior.

#### Structural Parameters

- `TOPOLOGY` *(str, required)*  
  Defines the underlying network topology.  
  **Allowed values:** `hcb`, `sql`  

- `BOND_TYPE` *(str, required)*  
  Specifies the bond type used to connect node and linker fragments.  
  **Allowed values:** `single`, `double`  
  **Constraint:** Must be consistent with the dummy atoms used in the input structures:  
  - `single` → **He**  
  - `double` → **Se**  
  **Note:** Mismatches may result in incorrect structure generation or runtime errors.

#### Output Parameters

- `COF_NAME` *(str, required)*  
  Unique identifier for the generated structure.  
  All output files and directories will be named using this value.  
  **Constraints:**  
  - Should be unique within the working directory.  
  - Avoid spaces and special characters for compatibility.  

#### Stacking Parameters

- `MODE` *(str, required)*  
  Defines the stacking mode(s) to be generated.  
  **Allowed values:**  
  - `incl` → inclined stacking  
  - `serr` → serrated stacking  
  - `both` → generate both stacking modes  

  **Behavior:**  
  Selecting `both` will generate both configurations sequentially and store them in the corresponding output structure.

In [2]:
TOPOLOGY = "hcb"
BOND_TYPE = "single" 
COF_NAME = "cof-1"
MODE = 'both'

#### MACE Head Configuration

- `MACE_HEAD` *(str, optional, default: ????? )*  
  Specifies the prediction head used by the `mace-mh-1` model.  

  **Allowed values (recommended):**  
  - `omat_pbe` -> D3 on, `dispersion_xc="pbe"`, `dispersion_cutoff=21.167088422553647`  
  - `matpes_r2scan` -> D3 on, `dispersion_xc="r2scan"`, `dispersion_cutoff=40.0`  
  - `omol` -> D3 off  
  - `spice_wB97M` -> D3 off

  **Reference:**  
  Model details: https://huggingface.co/mace-foundations/mace-mh-1  

  **Usage:**  
  Set `MACE_HEAD` in the subsequent cell to override the default value.

In [3]:
MACE_HEAD = "matpes_r2scan"

### Single-Layer COF Construction & Pre-Optimization

This step generates the single-layer COF structure and performs pre-optimization with `MaceOpt.optimize_cof`

#### Parameters

- `fmax` *(float, optional, default: `0.01`)*  
  Force convergence threshold for geometry optimization in `eV/Å`.

- `max_steps` *(int, optional, default: `???`)*  
  Maximum number of optimizer iterations per structure.  
  If convergence is not reached, the current structure is still written.

- `dtype` *(str, optional, default: `float64`)*  
  Numerical precision used during optimization.  
  **Allowed values:** `float32`, `float64`

- `head` *(str, optional, default: `???`)*  
  MACE head selection with fixed built-in settings.

- `device` *(str, optional, default: `cpu`)*  
  Compute device used for the calculation.  
  **Allowed values:** `cpu`, `cuda` *(if available)*

- `fix_z` *(bool, optional, default: `False`)*  
  If enabled, constrains atomic motion along the $z$-axis.

---

#### Input / Output  
- `input_path` *(str, required)*  
  Input CIF file path, typically:  
  `{COF_NAME}/1_{COF_NAME}_single_layer/{COF_NAME}_unopt.cif`

- `output_path` *(str, required)*  
  Output optimized CIF path, typically:  
  `{COF_NAME}/1_{COF_NAME}_single_layer/{COF_NAME}_preopt.cif`

In [ ]:
# Construction (Default)
builder = cl.BuildCOF2D()
builder.build(topo=TOPOLOGY, bond_type=BOND_TYPE, cof_name=COF_NAME)

# Pre-Optimization (Default)
preopt = cl.MaceOpt(head=MACE_HEAD,fix_z=True)
preopt.run_preopt(cof_name=COF_NAME)

In [ ]:
preopt = cl.MaceOpt(head=MACE_HEAD,fix_z=True)
preopt.run_preopt(cof_name=COF_NAME)

In [ ]:
# Construction (Configurable)
builder = cl.BuildCOF2D()
builder.build(
    topo=TOPOLOGY,
    input_node="0_node/HCB.xyz",
    input_linker="0_linker/Imidazol.xyz",
    output_folder=f"{COF_NAME}/1_{COF_NAME}_single_layer/",
    bond_type=BOND_TYPE,
    cof_name=COF_NAME,
 )

# Pre-Optimization (Configurable)
preopt = cl.MaceOpt(
    fmax=0.01,
    max_steps=500,
    dtype="float64",
    head=MACE_HEAD,
    device="cpu",
    fix_z=True,
 )
preopt.optimize_cof(
    input_path=f"{COF_NAME}/1_{COF_NAME}_single_layer/{COF_NAME}_unopt.cif",
    output_path=f"{COF_NAME}/1_{COF_NAME}_single_layer/{COF_NAME}_preopt.cif",
)

### ILD × ILS Structure Matrix Generation

This step generates stacked COF structures by systematically varying:

- **Interlayer Distance (ILD)** along the $z$-axis  
- **Interlayer Slipping (ILS)** within the plane  

The input structure is a single-layer (AA stacking with large interlayer distance).  
This step converts it into physically meaningful bulk stacking configurations.

---

#### Parameters

- `MODE` *(str, required)*  
  (see above under General Settings & Options)

- `ild_start`, `ild_end`, `ild_step` *(float, optional)*  
  Range of interlayer distances (in Å).  
  **Default:** `3.0 → 4.5` with step size `0.1`  

- `ils_length_start`, `ils_length_end`, `ils_length_step` *(float, optional)*  
  Range of in-plane slip magnitudes (in Å).  
  **Default:** `0 → max_AB_shift` with step size `1.0`  

- `ils_angle` *(float, optional)*  
  Direction of in-plane slipping in degrees.  
  **Default:** Automatically aligned with the AA → AB stacking direction  

- `print_shift` *(bool, optional, default: `False`)*  
  If enabled, prints the automatically computed maximum shift corresponding to AB stacking.  

  **Note:**  
  The default shift parameters are computed to interpolate between AA and AB stacking.  
  Manual modification is generally discouraged unless exploring non-standard stacking pathways.


---

#### Input / Output

- `input_cif` *(str, optional)*  
  Path to the input structure file.  
  **Default:**  
  `{COF_NAME}/1_{COF_NAME}_single_layer/{COF_NAME}_preopt.cif`

- `output_base_folder` *(str, optional)*  
  Base directory for generated structures.  
  **Default:**  
  `{COF_NAME}/2_{COF_NAME}_matrix/`
  Generated stacked structures organized by mode:  
  - `.../serr/` → serrated stacking configurations  
  - `.../incl/` → inclined stacking configurations  

These structures are intended for subsequent single-point energy evaluations and energy landscape analysis.

In [ ]:
# Defaults
matrix = cl.CreateMatrix()
matrix.run(cof_name=COF_NAME, topo=TOPOLOGY, mode=MODE)

In [ ]:
# Configurable
matrix = cl.CreateMatrix(
    ild_start=3.0,
    ild_end=4.0,
    ild_step=0.1,
    ils_length_start=0.0,
    ils_length_end=None,
    ils_length_step=1.0,
    ils_angle=None,
    print_shift=True,
 )
matrix.run(
    cof_name=COF_NAME,
    topo=TOPOLOGY,
    mode=MODE,
    input_cif=f"{COF_NAME}/1_{COF_NAME}_single_layer/{COF_NAME}_preopt.cif",
    output_base_folder=f"{COF_NAME}/2_{COF_NAME}_matrix/",
)

### MACE Single-Point Energy Evaluation

This step computes single-point energies for generated structures using `MaceSP` (no geometry optimization).

This provides a computationally efficient alternative to DFT-based evaluations, enabling rapid screening of stacking configurations.

> 🚨 **DFT Pre-Screening Strategy**
>
> Use this stage as a pre-screening step for DFT: sample the full ILS/ILD space at low cost with MACE, then select a smaller set for higher-accuracy DFT.

---

#### Parameters

- `dtype` *(str, optional, default: `float64`)*  
  Numerical precision used during evaluation.  
  **Allowed values:** `float32`, `float64`

- `head` *(str, optional, default: `MACE_HEAD`)*  
  MACE head with fixed built-in settings:  
  - `omat_pbe` -> D3 on, `dispersion_xc="pbe"`, `dispersion_cutoff=21.167088422553647`  
  - `matpes_r2scan` -> D3 on, `dispersion_xc="r2scan"`, `dispersion_cutoff=40.0`  
  - `omol` -> D3 off  
  - `spice_wB97M` -> D3 off

- `device` *(str, optional, default: `cpu`)*  
  Compute device used for evaluation.  
  **Allowed values:** `cpu`, `cuda` *(if available)*

---

#### Input / Output

- `input_folder` *(str, optional)*  
  Structure files (`.cif`) default read from:  
  `COF_NAME/2_{COF_NAME}_matrix/{serr|incl}/`

- `output_csv_dir` *(str, optional)* 
  CSV files containing single-point energies default written to:  
  `COF_NAME/3_{COF_NAME}_landscape/`  

  **File naming:**  
  `{COF_NAME}_sp_energies_{serr|incl}.csv`

---

In [ ]:
# Defaults
sp = cl.MaceSP(head=MACE_HEAD)
sp.run_mode(cof_name=COF_NAME, mode=MODE)

In [ ]:
# Configurable (all options)
sp = cl.MaceSP(
    device="cpu",
    dtype="float64",
    head=MACE_HEAD,
)
sp.run_mode(
    cof_name=COF_NAME,
    mode=MODE,
    input_folder=None,
    output_csv_dir=None,
)

### DFT Single-Point Input Generation (CRYSTAL23)

This step generates CRYSTAL23 `.d12` input files for all stacked structures.

The generated inputs must be executed externally (e.g., on an HPC system).  
After completion, the corresponding `.out` files should be placed back into the same directories as their respective `.d12` files.

**Default level of theory:** `HSEsol-3c / sol-mSVP`

---

#### Parameters

- `basisset` *(str, optional, default: `SOLDEF2MSVP`)*  
  Basis set used for all atoms in the system.

- `functional` *(str, optional, default: `HSESOL3C`)*  
  Exchange–correlation functional.

- `shrink` *(str, optional, default: `2 2 8`)*  
  Defines the Monkhorst–Pack $k$-point sampling grid (`n1 n2 n3`), where larger values increase Brillouin zone sampling accuracy at higher computational cost.

- `post_block` *(str or None, optional, default: `None`)*  
  Optional override for the full CRYSTAL23 input tail.  
  If `None`, a standard `BASISSET` / `DFT` / `SHRINK` block is automatically generated.

---

#### Input / Output

- `input_base_folder` *(str, optional)*  
  Structure files (`.cif`) default read from:  
  `COF_NAME/2_{COF_NAME}_matrix/{serr|incl}/`

- `output_base_folder` *(str, optional)*  
  CRYSTAL23 `.d12` input files default written to:  
  `COF_NAME/2_{COF_NAME}_matrix/dft_{serr|incl}/`

- **Post-processing:**  
  After external execution, place the resulting `.out` files in the same folders as their corresponding `.d12` files.

In [ ]:
# Defaults
sp = cl.CrystalSP()
sp.generate_input(cof_name=COF_NAME, mode=MODE)

In [ ]:
# Configurable
sp = cl.CrystalSP(
    basisset="SOLDEF2MSVP",
    functional="HSESOL3C",
    shrink="2 2 8",
    post_block=None,
 )
sp.generate_input(
    cof_name=COF_NAME,
    mode=MODE,
    input_base_folder=f"{COF_NAME}/2_{COF_NAME}_matrix/",
    output_base_folder=f"{COF_NAME}/2_{COF_NAME}_matrix/",
)

### CRYSTAL Energy Extraction

This step parses CRYSTAL23 `.out` files and extracts total energies for all generated structures.

Run this after all CRYSTAL jobs have completed and the `.out` files are placed alongside their corresponding `.d12` inputs.

---

#### Input / Output

- `input_base_folder` *(str, optional)*  
  CRYSTAL23 output files (`.out`) default located in:  
  `COF_NAME/2_{COF_NAME}_matrix/dft_{serr|incl}/`

- `output_base_folder` *(str, optional)*  
  CSV file(s) containing extracted energies default written to:  
  `COF_NAME/3_{COF_NAME}_landscape/`

  The data is structured for subsequent energy landscape analysis.

In [ ]:
# Defaults
sp = cl.CrystalSP()
sp.read_output(cof_name=COF_NAME, mode=MODE)

In [ ]:
# Configurable
sp = cl.CrystalSP()
sp.read_output(
    cof_name=COF_NAME,
    mode=MODE,
    output_base_folder=f"{COF_NAME}/3_{COF_NAME}_landscape",
    input_base_folder=f"{COF_NAME}/2_{COF_NAME}_matrix/",
)

### Potential Energy Landscape (PES)

This step constructs an approximate potential energy landscape by mapping total energies as a function of interlayer distance (ILD) and interlayer slipping (ILS).

The approach assumes that all structures are equivalent aside from their ILD/ILS values, providing a qualitative representation of relative stability across stacking configurations.  
The resulting landscape is used to identify candidate minima for subsequent refinement via full geometry optimization (e.g., MACE or DFT).

---

#### Parameters

- `colorscheme` *(str, optional, default: `viridis`)*  
  Matplotlib colormap used for visualization.  
  See: https://matplotlib.org/stable/users/explain/colors/colormaps.html

- `plot_mode` *(str, optional, default: `both`)*  
  Type of visualization to generate.  
  **Allowed values:** `heatmap`, `isolines`, `both`

- `rel_energy_max` *(float, optional)*  
  Upper cutoff for relative energies (in eV); values above this threshold are clipped in the plots.

- `show_minima_markers` *(bool, optional, default: `True`)*  
  If enabled, highlights minima markers on the PES.

- `minima_mode` *(str, optional, default: `global`)*  
  Controls minima handling:  
  - `global` -> mark only one global minimum  
  - `local` -> mark local minima (and the global minimum)

- `show_title_block` *(bool, optional, default: `True`)*  
  If enabled, draws the title and two header lines (stacking mode and level of theory).

- `show` *(bool, optional, default: `False`)*  
  If `True`, displays interactive plot windows.  
  Keep `False` for cluster/headless runs to avoid blocking or backend issues.

- `dft` *(bool, optional, default: `False`)*  
  If enabled, reads `_dft` CSVs and labels the plot title/level of theory as `DFT`.  
  Otherwise, the level of theory is displayed as `MACE-MH-1`.

---

#### Input / Output

- `input_folder` *(str, optional)*  
  Energy data (CSV) default read from:  
  `COF_NAME/3_{COF_NAME}_landscape/{COF_NAME}_sp_energies_{serr|incl}/`

- `output_folder` *(str, optional)*  
  PES plots default written to:  
  `COF_NAME/3_{COF_NAME}_landscape/pes_{COF_NAME}_{serr|incl}_{plot_mode}/`

In [ ]:
# Defaults
landscape = cl.Landscape()
landscape.run_mode(cof_name=COF_NAME, mode=MODE)

In [ ]:
# Configurable
landscape = cl.Landscape()
landscape.run_mode(
    cof_name=COF_NAME,
    mode=MODE,
    input_folder=f"{COF_NAME}/3_{COF_NAME}_landscape",
    output_folder=f"{COF_NAME}/3_{COF_NAME}_landscape",
    colorscheme="viridis",
    plot_mode="heatmap",
    rel_energy_max=1500,
    show_minima_markers=True,
    minima_mode="global",
    show_title_block=False,
    show=False,
    dft=False,
 )

### Additional PES Sampling Points

This allows manual inclusion of additional (ILD, ILS) points in the selection process.
The specified points can appended to the automatically detected minima for each stacking mode.

---

- `EXTRA_SERR` *(list[tuple[float, float]], optional)*  
  Additional points for serrated stacking.  
  Format: `[(ILD, ILS), ...]` in Å  

- `EXTRA_INCL` *(list[tuple[float, float]], optional)*  
  Additional points for inclined stacking.  
  Format: `[(ILD, ILS), ...]` in Å  

In [ ]:
EXTRA_SERR = [(3.6, 0.0)]
EXTRA_INCL = [(3.6, 0.0)]

### Structure Selection for Optimization

This step selects candidate structures based on the potential energy landscape and prepares them for downstream optimization.

Structures corresponding to automatically detected minima (and any user-defined ILD/ILS points) are copied into dedicated selection folders.

---

#### Parameters

- `selections_serr`  
  Additional (ILD, ILS) points for serrated stacking to include in the selection.  
  *(list[tuple[float, float]], optional)*

- `selections_incl`  
  Additional (ILD, ILS) points for inclined stacking to include in the selection.  
  *(list[tuple[float, float]], optional)*

- `include_autoselect`  
  If enabled, includes automatically detected minima from the PES.  
  *(bool, optional, default: `False`)*

- `autoselect_minima`  
  Which minima type is used by auto-selection.  
  *(str, optional, default: `global`)*  
  **Allowed values:** `global`, `local`

---

#### Input / Output

- `input_base` *(str, optional)*  
  Structure files (`.cif`) default read from:  
  `COF_NAME/2_{COF_NAME}_matrix/{serr|incl}/`

- `output_base` *(str, optional)*  
  Selected structures (`.cif`) default written to:  
  `COF_NAME/3_{COF_NAME}_landscape/selection/{serr|incl}/`

In [ ]:
# Defaults, auto-selected structures
selector = cl.SelectCofs()
selector.run_mode(
    cof_name=COF_NAME,
    mode=MODE,
    include_autoselect=True,
    autoselect_minima="global",
)

In [ ]:
# Defaults, manual selected structures
selector = cl.SelectCofs()
selector.run_mode(cof_name=COF_NAME, mode=MODE, selections_serr=EXTRA_SERR, selections_incl=EXTRA_INCL)

In [ ]:
# Configurable
selector = cl.SelectCofs()
selector.run_mode(
    cof_name=COF_NAME,
    mode=MODE,
    selections_serr=EXTRA_SERR,
    selections_incl=EXTRA_INCL,
    include_autoselect=True,
    autoselect_minima="local",
    input_base=f"{COF_NAME}/2_{COF_NAME}_matrix",
    output_base=f"{COF_NAME}/3_{COF_NAME}_landscape/selection",
 )

### MACE Geometry Optimization

This step performs geometry optimizations of selected stacking structures using `MaceOpt`.

`MaceOpt` uses ASE `FrechetCellFilter` + `LBFGS`, writes optimized CIF files, and can write a combined energy CSV (`{COF_NAME}_opt_energies_per_layer.csv`).

---

#### Parameters

- `fmax`  
  Force convergence threshold for geometry optimization in `eV/Å`.  
  *(float, optional, default: `0.01`)*

- `max_steps`  
  Maximum number of optimizer iterations per structure.  
  *(int, optional, default: `500`)*  
  If convergence is not reached, the current structure is written and optimization continues for remaining structures.

- `dtype`  
  Numerical precision used during optimization.  
  *(str, optional, default: `float64`)*  
  **Allowed values:** `float32`, `float64`

- `head`  
  MACE prediction head used for energy and force evaluation.  
  *(str, optional, default: `MACE_HEAD`)*  
 (see above under MACE Head Configuration)

- `device`  
  Compute device used for the calculation.  
  *(str, optional, default: `cpu`)*  
  **Allowed values:** `cpu`, `cuda` *(if available)*

- `fix_z`  
  Constrain atomic motion along the $z$-axis during relaxation when set to `True`.  
  *(bool, optional, default: `False`)*

- `save_opt_energies_csv`  
  If `True`, writes `COF_NAME/4_{COF_NAME}_optimization/{COF_NAME}_opt_energies_per_layer.csv`.  
  *(bool, optional, default: `True`)*

---

#### Input / Output

- `input_base` *(str, optional)*  
  Structure files (`.cif`) default read from:  
  `COF_NAME/3_{COF_NAME}_landscape/selection/{serr|incl}/`

- `output_base` *(str, optional)*  
  Optimized structures (`.cif`) and optional energies CSV are written to:  
  `COF_NAME/4_{COF_NAME}_optimization/`

In [ ]:
# Defaults
opt = cl.MaceOpt(head=MACE_HEAD, max_steps=2000)
opt.run_mode(cof_name=COF_NAME, mode=MODE)

In [ ]:
# Configurable
opt = cl.MaceOpt(
    fmax=0.01,
    max_steps=20,
    dtype="float64",
    head=MACE_HEAD,
    device="cpu",
    fix_z=False,
 )
opt.run_mode(
    cof_name=COF_NAME,
    mode=MODE,
    input_base=f"{COF_NAME}/3_{COF_NAME}_landscape/selection",
    output_base=f"{COF_NAME}/4_{COF_NAME}_optimization",
    save_opt_energies_csv=True,
)

### DFT Geometry Optimization Input Generation (CRYSTAL23)

This step generates CRYSTAL23 `.d12` input files for geometry optimization of the selected structures.

The generated inputs must be executed externally (e.g., on an HPC system).  
After completion, the corresponding `.out` files should be placed back into the same directories as their respective `.d12` files.

---

#### Parameters

- `basisset`  
  Basis set used for all atoms in the system.  
  *(str, optional, default: `SOLDEF2MSVP`)*

- `functional`  
  Exchange–correlation functional.  
  *(str, optional, default: `HSESOL3C`)*

- `shrink`  
  Defines the Monkhorst–Pack $k$-point sampling grid (`n1 n2 n3`), where larger values increase Brillouin zone sampling accuracy at higher computational cost.  
  *(str, optional, default: `2 2 8`)*

- `maxtradius`  
  Truncation radius for integral evaluation (in Å), controlling numerical accuracy of Coulomb and exchange contributions.  
  *(str, optional, default: `0.8`)*

- `post_block`  
  Optional override for the full CRYSTAL23 input tail.  
  *(str or None, optional, default: `None`)*  
  If `None`, a standard input block is automatically generated.

---

#### Input / Output

- `input_base` *(str, optional)*  
  Structure files (`.cif`) default read from:  
  `COF_NAME/3_{COF_NAME}_landscape/selection/{serr|incl}/`

- `output_base` *(str, optional)*  
  CRYSTAL23 `.d12` files default written to:  
  `COF_NAME/4_{COF_NAME}_optimization/dft_{serr|incl}/`


In [ ]:
# Defaults
opt = cl.CrystalOpt()
opt.generate_input(cof_name=COF_NAME, mode=MODE)

In [ ]:
# Configurable
opt = cl.CrystalOpt(
    basisset="SOLDEF2MSVP",
    functional="HSESOL3C",
    shrink="2 2 8",
    maxtradius="0.8",
    post_block=None,
 )

opt.generate_input(
    cof_name=COF_NAME,
    mode=MODE,
    input_base=f"{COF_NAME}/3_{COF_NAME}_landscape/selection",
    output_base=f"{COF_NAME}/4_{COF_NAME}_optimization",
)

### Extract CRYSTAL23 Optimized Structures

This step extracts optimized structures from CRYSTAL23 `.out` files, converts them to `.cif` format,
and reads total energies to generate CSV files with relative energies per layer for comparison across stacking configurations.

Run this after all CRYSTAL geometry optimization jobs have completed and the `.out` files are placed alongside their corresponding `.d12` inputs.

---

#### Input / Output

- `input_base_folder` *(str, optional)*  
  CRYSTAL23 output files (`.out`) default read from:  
  `COF_NAME/4_{COF_NAME}_optimization/dft_{serr|incl}/`

- `output_base_folder` *(str, optional)*  
  Extracted structures (`.cif`) and CSV files with relative energies per layer default written to:  
  `COF_NAME/4_{COF_NAME}_optimization/`

In [ ]:
# Defaults
opt = cl.CrystalOpt()
opt.extract_cif(cof_name=COF_NAME, mode=MODE)
opt.read_output(cof_name=COF_NAME, mode="incl")

In [ ]:
# Configurable
opt = cl.CrystalOpt()
opt.extract_cif(
    cof_name=COF_NAME,
    mode=MODE,
    output_base_folder=f"{COF_NAME}/4_{COF_NAME}_optimization",
    input_base_folder=f"{COF_NAME}/4_{COF_NAME}_optimization",
)
opt.read_output(
    cof_name=COF_NAME,
    mode=MODE,
    output_base_folder=f"{COF_NAME}/4_{COF_NAME}_optimization",
    input_base_folder=f"{COF_NAME}/4_{COF_NAME}_optimization",
)

### Analysis & Visualization

This step analyzes optimized structures by computing interlayer distance (ILD) and interlayer slipping (ILS), and writing a summary CSV for comparison across stacking configurations.

In addition, structures can be visualized using an interactive viewer with configurable supercell sizes.

---

#### Parameters

- `dft`  
  If enabled, uses DFT-optimized structures instead of MACE-optimized ones.  
  *(bool, optional, default: `False`)*

- `print_values`  
  If enabled, prints computed ILD/ILS values to the console.  
  *(bool, optional, default: `True`)*

- `add_unit_cell`  
  If enabled, displays the unit cell in the visualization.  
  *(bool, optional, default: `True`)*

- `supercell_size_serr`  
  Supercell size for serrated structures in the viewer.  
  *(tuple[int, int, int], optional, default: `(2, 2, 1)`)*

- `supercell_size_incl`  
  Supercell size for inclined structures in the viewer.  
  *(tuple[int, int, int], optional, default: `(2, 2, 2)`)*

---

#### Input / Output

- `input_base` *(str, optional)*  
  Structures (`.cif`) default read from:  
  - `COF_NAME/4_{COF_NAME}_optimization/{serr|incl}/` *(dft=False)*  
  - `COF_NAME/4_{COF_NAME}_optimization/dft_{serr|incl}/` *(dft=True)*

- `output_base` *(str, optional)*  
  Analysis results (CSV) default written to:  
  `COF_NAME/5_{COF_NAME}_analysis/`  

  **Files:**  
  - `final_structures.csv` *(dft=False)*  
  - `final_structures_dft.csv` *(dft=True)*

In [ ]:
# Defaults
analyze = cl.analyze
analyze(cof_name=COF_NAME, mode="incl")

In [ ]:
# Configurable
analyze = cl.analyze
analyze(
    cof_name=COF_NAME,
    mode="incl",
    input_base=f"{COF_NAME}/4_{COF_NAME}_optimization",
    output_base=f"{COF_NAME}/5_{COF_NAME}_analysis",
    dft=True,
    print_values=True,
)

In [ ]:
# Defaults
visualize = cl.visualize_cof
visualize(cof_name=COF_NAME, mode=MODE, dft=True)

In [ ]:
# Configurable
visualize = cl.visualize_cof
visualize(
    cof_name=COF_NAME,
    mode=MODE,
    input_base=f"{COF_NAME}/4_{COF_NAME}_optimization",
    dft=True,
    add_unit_cell=True,
    supercell_size_serr=(2, 2, 1),
    supercell_size_incl=(2, 2, 2),
)

### PXRD Pattern Generation

This step simulates PXRD patterns from optimized CIF structures and writes per-structure `.xy` files.

---

#### Parameters

- `wavelength`  
  X-ray source line used for simulation.  
  *(str, optional, default: `CuKa`)*  
  **Allowed values:** `CuKa`, `MoKa`, `CrKa`, `FeKa`, `CoKa`, `AgKa`  
  Choose this to match your instrument/source.

- `two_theta_range`  
  Angular simulation window in degrees for generated `.xy` data.  
  *(tuple[float, float], optional, default: `(1.5, 60.0)`)*

- `dft`  
  If enabled, uses DFT-optimized structures instead of MACE-optimized ones.  
  *(bool, optional, default: `False`)*

---

#### Input / Output

- `input_folder` *(str, optional)*  
  Structure files (`.cif`) default read from:  
  - `COF_NAME/4_{COF_NAME}_optimization/{serr|incl}/` *(dft=False)*  
  - `COF_NAME/4_{COF_NAME}_optimization/dft_{serr|incl}/` *(dft=True)*

- `output_folder` *(str, optional)*  
  PXRD data (`.xy`) default written to:  
  - `COF_NAME/5_{COF_NAME}_analysis/pxrd_xy/{serr|incl}/` *(dft=False)*  
  - `COF_NAME/5_{COF_NAME}_analysis/pxrd_xy_dft/{serr|incl}/` *(dft=True)*

In [ ]:
# Run: Defaults

pxrd = cl.Pxrd(wavelength="CuKa", two_theta_range=(1.5, 60.0))
pxrd.run(cof_name=COF_NAME, mode=MODE)

In [ ]:
# Run: Configurable

pxrd = cl.Pxrd(
    wavelength="CuKa",
    two_theta_range=(1.5, 60.0),
)

pxrd.run(
    cof_name=COF_NAME,
    mode=MODE,
    dft=True,
    input_folder=f"{COF_NAME}/4_{COF_NAME}_optimization",
    output_folder=f"{COF_NAME}/5_{COF_NAME}_analysis/pxrd_xy_dft",
)

### PXRD Plot

This step reads `.xy` files and creates stacked PXRD plots.

#### Parameters

- `xlim`  
  X-axis limits used for plotting in 2-theta degrees.  
  *(tuple[float, float], optional, default: `(1.5, 60.0)`)*

- `show`  
  If enabled, displays generated PXRD plots in the notebook.  
  *(bool, optional, default: `True`)*

- `dft`  
  Controls default naming/path behavior (adds `_dft` suffix in output filenames).  
  *(bool, optional, default: `False`)*

#### Input / Output

- `xy_folder` *(str, optional)*  
  PXRD data (`.xy`) default read from:  
  - `COF_NAME/5_{COF_NAME}_analysis/pxrd_xy/{serr|incl}/` *(dft=False)*  
  - `COF_NAME/5_{COF_NAME}_analysis/pxrd_xy_dft/{serr|incl}/` *(dft=True)*

- `output_folder` *(str, optional)*  
  PXRD plots (`.png`) default written to:  
  `COF_NAME/5_{COF_NAME}_analysis/`

  **Files:**  
  - `pxrd_stacked_{serr|incl}.png`  
  - `pxrd_stacked_{serr|incl}_dft.png` *(dft=True)*

---

#### Notes

- PXRD patterns are simulated from optimized structures without additional structural modification.  
- Plotting uses stacked representations to facilitate comparison between stacking configurations.

In [ ]:
# Plot: Defaults

pxrd = cl.Pxrd()
pxrd.plot(cof_name=COF_NAME, mode="incl", xlim=(1.5, 60.0),)

In [ ]:
# Plot: Configurable

pxrd = cl.Pxrd()
pxrd.plot(
    cof_name=COF_NAME,
    mode=MODE,
    dft=True,
    xy_folder=f"{COF_NAME}/5_{COF_NAME}_analysis/pxrd_xy_dft",
    output_folder=f"{COF_NAME}/5_{COF_NAME}_analysis",
    xlim=(1.5, 60.0),
    show=True,
)